# Benchmark: Brain Aging Spatial Atlas (MERFISH)

Two conditions:
- **Control**: `BrainAgingSpatialAtlas_MERFISH.h5ad` — 378,918 cells × 374 genes
- **LPS-treated**: `BrainAgingSpatialAtlas_MERFISH_LPS.h5ad` — 345,934 cells × 374 genes

Both have spatial coordinates (`X_spatial_coords`) and UMAP embeddings (`X_umap`).

In [6]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)

DATA_DIR = '/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler/benchmark/'

In [ ]:
adata_ctrl = ad.read_h5ad(DATA_DIR + 'BrainAgingSpatialAtlas_MERFISH.h5ad')
adata_lps  = ad.read_h5ad(DATA_DIR + 'BrainAgingSpatialAtlas_MERFISH_LPS.h5ad')

adata_ctrl.obs['condition'] = 'control'
adata_lps.obs['condition']  = 'LPS'

print('Control:', adata_ctrl)
print('\nLPS:    ', adata_lps)

In [ ]:
for label, adata in [('Control', adata_ctrl), ('LPS', adata_lps)]:
    print(f'\n--- {label} ---')
    print('n_obs:', adata.n_obs, '  n_vars:', adata.n_vars)
    print('Slices:', adata.obs['slice'].unique().tolist())
    print('Ages:  ', sorted(adata.obs['age'].unique().tolist()))
    print('Cell types (n):', adata.obs['cell_type_annot'].nunique())
    print(adata.obs['cell_type_annot'].value_counts().to_string())


--- Control ---
n_obs: 378918   n_vars: 374
Slices: ['0', '1', '2']
Ages:   ['24wk', '4wk', '90wk']
Cell types (n): 13
cell_type_annot
ExN       107009
Olig       71592
MSN        63455
Astro      37480
Endo       30038
InN        27997
Micro      16514
Peri        9675
OPC         9637
Vlmc        2791
Epen        1489
Macro       1031
T cell       210

--- LPS ---
n_obs: 345934   n_vars: 374
Slices: ['0', '1', '2']
Ages:   ['24wk', '4wk', '90wk']
Cell types (n): 13
cell_type_annot
ExN       97214
MSN       63134
Olig      56766
Astro     37530
Endo      30604
InN       25248
Micro     13620
Peri       8859
OPC        8274
Vlmc       2373
Epen       1362
Macro       743
T cell      207


## Spatial maps

One panel per tissue slice, colored by cell-type annotation. Subsampled to 30k cells/slice for speed.

In [ ]:
def plot_spatial_slice(adata, title, color='cell_type_annot', slice_id='0',
                       point_size=2, subsample=None):
    """All donors combined for one anatomical slice — the assembled atlas view."""
    sub = adata[adata.obs['slice'] == slice_id]
    if subsample and sub.n_obs > subsample:
        idx = np.random.choice(sub.n_obs, subsample, replace=False)
        sub = sub[idx]

    cats = sub.obs[color].astype('category').cat.categories
    cmap = plt.colormaps['tab20'].resampled(len(cats))
    col_dict = {c: cmap(i) for i, c in enumerate(cats)}

    cx = sub.obs['center_x'].values.astype(float)
    cy = sub.obs['center_y'].values.astype(float)
    colors = [col_dict[c] for c in sub.obs[color].astype(str)]

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(cx, cy, c=colors, s=point_size, linewidths=0, edgecolors='none')
    ax.invert_yaxis()
    ax.set_aspect('equal', 'box')
    ax.axis('off')

    legend_patches = [mpatches.Patch(color=col_dict[c], label=c) for c in cats]
    ax.legend(handles=legend_patches, fontsize=7, ncol=2,
              loc='lower right', frameon=False, markerscale=3)
    ax.set_title(f'{title}  (slice {slice_id}, n={sub.n_obs:,})', fontsize=12)
    plt.tight_layout()
    plt.show()


def plot_spatial_per_donor(adata, title, color='cell_type_annot', group_by='donor_id',
                           slice_filter=None, point_size=5, ncols=4):
    """One panel per donor — for inspecting individual tissue sections."""
    sub = adata[adata.obs['slice'] == slice_filter] if slice_filter is not None else adata
    groups = sorted(sub.obs[group_by].unique())
    n = len(groups)
    nrows = int(np.ceil(n / ncols))

    cats = sub.obs[color].astype('category').cat.categories
    cmap = plt.colormaps['tab20'].resampled(len(cats))
    col_dict = {c: cmap(i) for i, c in enumerate(cats)}

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
    axes = np.array(axes).flatten()

    for ax, g in zip(axes, groups):
        gsub = sub[sub.obs[group_by] == g]
        cx = gsub.obs['center_x'].values.astype(float)
        cy = gsub.obs['center_y'].values.astype(float)
        colors = [col_dict[c] for c in gsub.obs[color].astype(str)]
        ax.scatter(cx, cy, c=colors, s=point_size, linewidths=0, edgecolors='none')
        ax.invert_yaxis()
        ax.set_aspect('equal', 'box')
        label = str(g).replace('MsBrainAgingSpatialDonor_', 'Donor ')
        ax.set_title(label, fontsize=8)
        ax.axis('off')

    for ax in axes[n:]:
        ax.set_visible(False)

    legend_patches = [mpatches.Patch(color=col_dict[c], label=c) for c in cats]
    fig.legend(handles=legend_patches, loc='lower center', ncol=5,
               fontsize=7, bbox_to_anchor=(0.5, -0.02), frameon=False)
    slice_str = f'  (slice {slice_filter})' if slice_filter is not None else ''
    fig.suptitle(title + slice_str, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Combined atlas view — all donors assembled for each anatomical slice
for sl in ['0', '1', '2']:
    plot_spatial_slice(adata_ctrl, 'Control — cell type', slice_id=sl)

In [ ]:
for sl in ['0', '1', '2']:
    plot_spatial_slice(adata_lps, 'LPS — cell type', slice_id=sl)

In [ ]:
plot_spatial_slice(adata_ctrl, 'Control — age', color='age', slice_id='0')
plot_spatial_slice(adata_lps,  'LPS — age',     color='age', slice_id='0')

# Per-donor view for closer inspection (optional — change slice_filter as needed)
plot_spatial_per_donor(adata_ctrl, 'Control — per donor', slice_filter='0')

## UMAP embeddings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (label, adata) in zip(axes, [('Control', adata_ctrl), ('LPS', adata_lps)]):
    cats = adata.obs['cell_type_annot'].astype('category').cat.categories
    cmap = plt.cm.get_cmap('tab20', len(cats))
    col_dict = {c: cmap(i) for i, c in enumerate(cats)}

    idx = np.random.choice(adata.n_obs, min(50_000, adata.n_obs), replace=False)
    sub = adata[idx]
    umap   = sub.obsm['X_umap']
    colors = [col_dict[c] for c in sub.obs['cell_type_annot'].astype(str)]

    ax.scatter(umap[:, 0], umap[:, 1], c=colors, s=0.3, linewidths=0)
    ax.set_title(f'{label} — UMAP (cell type)', fontsize=11)
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')

    legend_patches = [mpatches.Patch(color=col_dict[c], label=c) for c in cats]
    ax.legend(handles=legend_patches, fontsize=6, markerscale=3,
              loc='upper right', frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# UMAP coloured by age — compare aging pattern across conditions
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

all_ages = sorted(set(adata_ctrl.obs['age'].unique()) | set(adata_lps.obs['age'].unique()))
age_cmap = plt.cm.get_cmap('plasma', len(all_ages))
age_col  = {a: age_cmap(i) for i, a in enumerate(all_ages)}

for ax, (label, adata) in zip(axes, [('Control', adata_ctrl), ('LPS', adata_lps)]):
    idx = np.random.choice(adata.n_obs, min(50_000, adata.n_obs), replace=False)
    sub = adata[idx]
    umap   = sub.obsm['X_umap']
    colors = [age_col[a] for a in sub.obs['age']]
    ax.scatter(umap[:, 0], umap[:, 1], c=colors, s=0.3, linewidths=0)
    ax.set_title(f'{label} — UMAP (age)', fontsize=11)
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
    legend_patches = [mpatches.Patch(color=age_col[a], label=a) for a in all_ages]
    ax.legend(handles=legend_patches, fontsize=7, loc='upper right', frameon=False)

plt.tight_layout()
plt.show()

## Cell-type composition comparison

In [ ]:
ctrl_frac = adata_ctrl.obs['cell_type_annot'].value_counts(normalize=True).rename('control')
lps_frac  = adata_lps.obs['cell_type_annot'].value_counts(normalize=True).rename('LPS')
comp = pd.concat([ctrl_frac, lps_frac], axis=1).fillna(0).sort_values('control', ascending=False)

comp.plot(kind='bar', figsize=(14, 4), width=0.7)
plt.ylabel('Fraction of cells')
plt.title('Cell-type composition: control vs LPS')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()